# Stage 6 & 7: Algorithm 4 — Multi-Layer Perceptron (MLP Neural Network)

**Owner:** Member D  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/mlp_tuned.pkl`  

### Scope
This notebook implements the Multi-Layer Perceptron (MLP) neural network model family for late delivery risk prediction.
It trains a baseline MLP network, performs Stratified K-Fold hyperparameter tuning on `X_train` (tuning hidden layer architecture and L2 penalty `alpha`), evaluates performance on `X_val`, and exports the tuned model artifact.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report
)

FINAL_DIR = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}')


In [ ]:
# ── Baseline MLP Classifier ─────────────────────────────────────────────────
mlp_base = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, early_stopping=True, random_state=42)
mlp_base.fit(X_train, y_train)
base_probs = mlp_base.predict_proba(X_val)[:, 1]
print(f'Baseline MLP Val ROC-AUC : {roc_auc_score(y_val, base_probs):.4f}')
print(f'Baseline MLP Val PR-AUC  : {average_precision_score(y_val, base_probs):.4f}')


In [ ]:
# ── Stratified GridSearchCV ─────────────────────────────────────────────────
param_grid = {
    'hidden_layer_sizes': [(64, 32), (128, 64)],
    'alpha': [0.001, 0.01],
    'activation': ['relu']
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(
    MLPClassifier(max_iter=100, early_stopping=True, random_state=42),
    param_grid, scoring='average_precision', cv=cv, n_jobs=-1
)
grid.fit(X_train, y_train)

print('Best Hyperparameters:', grid.best_params_)
print(f'Best Stratified CV PR-AUC: {grid.best_score_:.4f}')

best_mlp = grid.best_estimator_
joblib.dump(best_mlp, MODELS_DIR / 'mlp_tuned.pkl')
print('Saved model artifact to models/mlp_tuned.pkl')
